Step 2

In [ ]:
"""
Espoo District Heating Network Optimization
"""


import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import dhnx
import os
import pandas as pd


import os

# Adjust this path to match your actual project structure
base_dir = "c:/Users/matis/Desktop/KTH/Practical Optimisation/Tutorial 1/Network_optmisation/Project__DH"
twn_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "twn_data_step2")
invest_data_path = os.path.join(base_dir, "DHNx_files", "Espoo", "invest_data_step2")


print('='*60)
print('ESPOO DHN - INVESTMENT OPTIMIZATION')
print('='*60)

# Load network
print('\n[1/3] Loading network...')
network = dhnx.network.ThermalNetwork()
network = network.from_csv_folder(twn_data_path)
invest_opt = dhnx.input_output.load_invest_options(invest_data_path)

print(f'  - Producers: {len(network.components.producers)}')
print(f'  - Consumers: {len(network.components.consumers)}')
print(f'  - Pipe segments: {len(network.components.pipes)}')

# Plot initial network topology
print('\nPlotting initial network...')
plt.figure(figsize=(10, 10))
static_map_initial = dhnx.plotting.StaticMap(network)
static_map_initial.draw(background_map=False)
plt.scatter(network.components.producers['lon'], network.components.producers['lat'],
            color='tab:red', label='producers', zorder=2.5, s=150)
plt.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
            color='tab:green', label='consumers', zorder=2.5, s=100)
plt.scatter(network.components.forks['lon'], network.components.forks['lat'],
            color='tab:grey', label='forks', zorder=2.5, s=50)
plt.title('Espoo DHN - Initial Network Topology', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.xlim(24.14, 24.20)
plt.tight_layout()
plt.savefig('Outputs/network_initial_step2.png', dpi=150, bbox_inches='tight')
plt.close()


# Run optimization with MIPGAP for faster solving
print('\n[2/3] Running optimization with MIPGAP=0.05 (5% optimality gap)...')
# GLPK mipgap must be nested in options dict within solve_kw
network.optimize_investment(invest_options=invest_opt, solver='glpk', 
                           solve_kw={'tee': True, 'options': {'mipgap': 0.05}})


# Get results
results = network.results.optimization['components']['pipes']
results.to_csv("Outputs/optimization_results.csv")

# Summary
print('\n[3/3] Results:')
print('-'*60)
objective = network.results.optimization['oemof_meta']['objective']
print(f'Total Cost: {objective:,.0f} EUR')

# Pipe types used
active_pipes = results[results['capacity'] > 0.001]
pipe_counts = active_pipes['hp_type'].value_counts()
print(f'\nPipes installed:')
for pipe_type, count in pipe_counts.items():
    total_cap = active_pipes[active_pipes['hp_type'] == pipe_type]['capacity'].sum()
    print(f'  {pipe_type}: {count} segments ({total_cap:.1f} kW)')

# Plot optimized network with pipe types color-coded
print('\nCreating optimized network plot with pipe types...')

# Define colors for different pipe types
pipe_colors = {
    'DN30': 'blue',
    'DN50': 'green', 
    'DN65': 'orange',
    'DN80': 'red',
    'DN100': 'purple'
}

# Create figure with equal dimensions
fig, ax = plt.subplots(figsize=(10, 10))

# Create a mapping of all node IDs to coordinates
node_coords = {}

# Add all nodes with their coordinates
for idx, row in network.components.producers.iterrows():
    node_coords[f'producers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.consumers.iterrows():
    node_coords[f'consumers-{idx}'] = (row['lon'], row['lat'])

for idx, row in network.components.forks.iterrows():
    node_coords[f'forks-{idx}'] = (row['lon'], row['lat'])

# Draw pipes grouped by type
from matplotlib.lines import Line2D

for pipe_type in sorted(active_pipes['hp_type'].unique()):
    pipes_of_type = active_pipes[active_pipes['hp_type'] == pipe_type]
    color = pipe_colors.get(pipe_type, 'black')
    
    for idx, pipe in pipes_of_type.iterrows():
        from_node = pipe['from_node']
        to_node = pipe['to_node']
        
        # Get coordinates
        if from_node in node_coords and to_node in node_coords:
            x_coords = [node_coords[from_node][0], node_coords[to_node][0]]
            y_coords = [node_coords[from_node][1], node_coords[to_node][1]]
            
            ax.plot(x_coords, y_coords, color=color, linewidth=3, alpha=0.8, zorder=1)

# Add nodes on top
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
          color='tab:green', s=100, edgecolors='black', linewidths=1, zorder=3, label='Consumers')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
          color='tab:red', s=150, edgecolors='black', linewidths=1, zorder=3, label='Producers')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
          color='tab:grey', s=50, edgecolors='black', linewidths=0.5, zorder=2, label='Forks')

# Create custom legend for pipe types
pipe_legend_elements = [Line2D([0], [0], color=pipe_colors.get(pt, 'black'), linewidth=3, 
                               label=f'{pt} ({pipe_counts[pt]} seg)')
                       for pt in sorted(pipe_counts.index)]

# Get handles and labels from existing legend
handles, labels = ax.get_legend_handles_labels()

# Combine pipe types and node types in legend
all_handles = pipe_legend_elements + handles
ax.legend(handles=all_handles, loc='best', fontsize=9)

ax.set_title('Espoo DHN - Optimized Network with Pipe Types', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(24.14, 24.20)
plt.tight_layout()
plt.savefig('Outputs/network_optimized_step2.png', dpi=150, bbox_inches='tight')
plt.close()


print('\nGenerated files:')
print('  - Outputs/network_initial_step2.png')
print('  - Outputs/network_optimized_step2.png')
print('  - Outputs/optimization_results.csv')